# Rank ablation: r = 8, 16 and 64 on the same 1,000 rows

Three short runs of `training/sft.py`, identical except for the LoRA rank: the same seeded 1,000-row subset of
the train split, the same learning rate, steps and seed. Alpha moves with r (8/8, 16/16, 64/64) so the scale
alpha / r stays at 1 and the runs differ in capacity only. Each run measures validation and held-out-intent
loss every 10 steps and generates the probes; nothing is pushed to the Hub.

**Before running (Kaggle):** GPU **T4 x2**, Internet **On**, secret **`HF_TOKEN`** switched on (the dataset is
private). **Save & Run All** is fine. Three runs of 63 steps each.

**After:** Output → download **`ablation-runs.zip`**, unzip it into the repo's `training/runs/`, and paste the
table from cell 4 into `training/ABLATIONS.md`.

In [ ]:
# 1. Settings and secrets.
RANKS = [8, 16, 64]
COMMON = ["train_rows=1000", "eval_steps=10", "save_steps=1000"]   # 63 steps; no mid-run checkpoints

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# 2. Clone the repo and install the pinned training stack.
import subprocess, sys
REPO_DIR = os.path.abspath("customer-support-SLM")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
%pip install -q -r requirements-gpu.txt
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. The three runs, one after another. Each is a fresh process, so GPU memory starts empty every time.
from training.sft import load_config, run_name
RUNS = []
for r in RANKS:
    sets = COMMON + [f"r={r}", f"lora_alpha={r}"]
    RUNS.append(run_name(load_config("training/configs/sft1.yaml", sets)))
    SETS = " ".join(sets)
    print(f"\n==================== {RUNS[-1]}")
    !python training/sft.py --config training/configs/sft1.yaml --set {SETS}

In [ ]:
# 4. The table for training/ABLATIONS.md, and the validation curves side by side.
import json, matplotlib.pyplot as plt
OUTS = [f"outputs/{n}" for n in RUNS]
OUT_ARGS = " ".join(OUTS)
!python training/ablation_table.py {OUT_ARGS} | tee /kaggle/working/ablation-table.md
fig, ax = plt.subplots(figsize=(8, 4))
for out, r in zip(OUTS, RANKS):
    hist = json.load(open(f"{out}/log_history.json"))
    for key, style in [("eval_validation_loss", "-"), ("eval_unseen_loss", "--")]:
        pts = [(h["step"], h[key]) for h in hist if key in h]
        ax.plot(*zip(*pts), style, marker="o", label=f"r={r} {'validation' if 'validation' in key else 'held-out'}")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.show()

In [ ]:
# 5. Every probe's final reply, the three ranks next to each other.
import pandas as pd
final = []
for out, r in zip(OUTS, RANKS):
    p = pd.read_json(f"{out}/probes.jsonl", lines=True)
    final.append(p[p.step == p.step.max()].assign(r=r))
final = pd.concat(final)
for prompt, g in final.groupby("prompt", sort=False):
    print(f"\n=== {prompt!r}")
    for _, row in g.iterrows():
        print(f"  r={row.r:<3} ({row.words}w) {row.reply}")

In [ ]:
# 6. The runs' small files for the repo (no weights): download ablation-runs.zip from Output.
import shutil, tempfile
tmp = tempfile.mkdtemp()
for n in RUNS:
    shutil.copytree(f"outputs/{n}", f"{tmp}/{n}", ignore=shutil.ignore_patterns(
        "checkpoint-*", "*.safetensors", "*.bin", "tokenizer*", "*.jinja", "*config.json"))
shutil.make_archive("/kaggle/working/ablation-runs", "zip", tmp)
print(sorted(os.listdir(tmp)))